# Análisis Multi-Especie de Ocurrencias de Biodiversidad en GBIF (España y Portugal)

Este Jupyter Notebook automatiza la consulta, limpieza, enriquecimiento, exportación tabular y visualización geoespacial de registros de presencia de biodiversidad desde la **API pública de GBIF** (estándar **Darwin Core**) para una **lista configurable de especies** en **España (`ES`)** y **Portugal (`PT`)**.

### Flujo de trabajo del pipeline:
1. **Resolución taxonómica dinámica (`/v1/species/match`)**: Convierte cada nombre científico de la lista en su identificador único `usageKey` (`taxonKey`) y verifica su rango taxonómico y nivel de coincidencia.
2. **Extracción paginada (`/v1/occurrence/search`)**: Descarga registros georreferenciados (`hasCoordinate=True`) por especie y país, controlando el límite máximo de registros por especie para evitar bloqueos en taxones muy abundantes.
3. **Normalización Darwin Core y limpieza de URLs**: Extrae los campos clave, construye el enlace directo a la ficha de GBIF (`https://www.gbif.org/occurrence/{gbifID}`), formatea correctamente múltiples URLs multimedia sin generar cadenas malformadas ni valores `NaN` inconsistentes y filtra coordenadas inválidas.
4. **Exportación individual y consolidada**: Guarda un archivo `.csv` independiente por especie y un archivo maestro consolidado (`gbif_multiespecie_es_pt.csv`), además de generar un resumen comparativo.
5. **Cartografía interactiva sin API Key (`folium`)**: Genera mapas HTML individuales con `MarkerCluster` y un mapa comparativo multicapa (`LayerControl`) usando teselas abiertas (OpenStreetMap y Esri World Imagery).

In [1]:
import re
import time
from pathlib import Path
from typing import Any

import folium
from folium.plugins import MarkerCluster
import pandas as pd
import truststore

# Netskope, en la red de la Junta, sustituye el certificado de GBIF.
# El almacén de Windows incluye esa CA; la verificación TLS sigue activa.
truststore.inject_into_ssl()
import requests

# ==============================================================================
# PARÁMETROS DE CONFIGURACIÓN DEL PIPELINE
# ==============================================================================
ESPECIES_OBJETIVO = [
    "Canis lupus",
    "Lynx pardinus",
    "Ursus arctos",
    "Gypaetus barbatus",
]

PAISES = ["ES", "PT"]
MAX_REGISTROS_POR_ESPECIE_PAIS = 600  # Límite por país para evitar descargas masivas
TAMANO_PAGINA = 300                   # Máximo permitido por petición en GBIF (300)
CARPETA_SALIDA = (Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent) / "salidas_gbif_multiespecie"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

HEADERS_HTTP = {
    "User-Agent": "GBIF-MultiSpecies-Biodiversity-Pipeline/1.0 (Python requests)"
}

COLUMNAS_DARWIN_CORE = [
    "especie_buscada",
    "gbifID",
    "url_ocurrencia_gbif",
    "scientificName",
    "acceptedScientificName",
    "taxonRank",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
    "countryCode",
    "stateProvince",
    "locality",
    "decimalLatitude",
    "decimalLongitude",
    "coordinateUncertaintyInMeters",
    "eventDate",
    "year",
    "month",
    "day",
    "basisOfRecord",
    "institutionCode",
    "collectionCode",
    "datasetKey",
    "occurrenceID",
    "recordedBy",
    "informationWithheld",
    "media_urls",
]

print(f"Configuración cargada. Especies a procesar: {len(ESPECIES_OBJETIVO)}")
print(f"Directorio de salida: {CARPETA_SALIDA.resolve()}")

Configuración cargada. Especies a procesar: 4
Directorio de salida: C:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie


## 1. Resolución Taxonómica Dinámica (`/v1/species/match`)

Antes de consultar ocurrencias, cada nombre científico se valida contra el *Backbone Taxonomy* de GBIF. Esto garantiza obtener el `usageKey` exacto y detectar si el nombre corresponde a una especie, subespecie o sinonimia.

In [2]:
def normalizar_slug(nombre: str) -> str:
    """Convierte un nombre científico en un identificador seguro para nombres de archivo."""
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", nombre.strip().lower()).strip("_")
    return slug


def resolver_taxon_gbif(nombre_cientifico: str) -> dict[str, Any]:
    """Consulta /v1/species/match para obtener el usageKey y metadatos taxonómicos."""
    url = "https://api.gbif.org/v1/species/match"
    params = {"name": nombre_cientifico, "strict": False}
    
    resp = requests.get(url, params=params, headers=HEADERS_HTTP, timeout=20)
    resp.raise_for_status()
    data = resp.json()
    
    return {
        "especie_buscada": nombre_cientifico,
        "slug": normalizar_slug(nombre_cientifico),
        "usageKey": data.get("usageKey"),
        "scientificName": data.get("scientificName"),
        "canonicalName": data.get("canonicalName"),
        "rank": data.get("rank"),
        "status": data.get("status"),
        "matchType": data.get("matchType"),
        "confidence": data.get("confidence"),
    }


catalogo_taxones = []
for especie in ESPECIES_OBJETIVO:
    info = resolver_taxon_gbif(especie)
    catalogo_taxones.append(info)

df_catalogo = pd.DataFrame(catalogo_taxones)
display(df_catalogo)

,especie_buscada,slug,usageKey,scientificName,canonicalName,rank,status,matchType,confidence
0,Canis lupus,canis_lupus,5219173,"Canis lupus Linnaeus, 1758",Canis lupus,SPECIES,ACCEPTED,EXACT,99
1,Lynx pardinus,lynx_pardinus,2435261,"Lynx pardinus (Temminck, 1827)",Lynx pardinus,SPECIES,ACCEPTED,EXACT,99
2,Ursus arctos,ursus_arctos,2433433,"Ursus arctos Linnaeus, 1758",Ursus arctos,SPECIES,ACCEPTED,EXACT,99
3,Gypaetus barbatus,gypaetus_barbatus,2480649,"Gypaetus barbatus (Linnaeus, 1758)",Gypaetus barbatus,SPECIES,ACCEPTED,EXACT,99


## 2. Extracción Paginada y Limpieza bajo Estándar Darwin Core

Para cada especie resuelta con `usageKey` válido:
* Se consulta `/v1/occurrence/search` filtrando por `taxonKey`, `country` (`ES` y `PT`) y `hasCoordinate=True`.
* Se extraen de forma segura los enlaces multimedia del array `media` (evitando comas sueltas, corchetes o cadenas malformadas al exportar a CSV).
* Se añade la columna `url_ocurrencia_gbif` apuntando directamente a `https://www.gbif.org/occurrence/{gbifID}`.

In [3]:
def extraer_urls_multimedia(registro: dict[str, Any]) -> str:
    """Extrae y limpia las URLs multimedia de un registro GBIF separándolas por ' | '."""
    media_lista = registro.get("media")
    if not isinstance(media_lista, list) or not media_lista:
        return ""
    
    urls_limpias = []
    for item in media_lista:
        if isinstance(item, dict):
            url = item.get("identifier") or item.get("references")
            if isinstance(url, str) and url.strip().startswith(("http://", "https://")):
                url_norm = url.strip()
                if url_norm not in urls_limpias:
                    urls_limpias.append(url_norm)
    return " | ".join(urls_limpias)


def descargar_ocurrencias_taxon(
    taxon_info: dict[str, Any],
    paises: list[str] = PAISES,
    max_por_pais: int = MAX_REGISTROS_POR_ESPECIE_PAIS,
) -> pd.DataFrame:
    """Descarga ocurrencias georreferenciadas de GBIF para un taxón y lista de países."""
    usage_key = taxon_info.get("usageKey")
    especie_buscada = taxon_info["especie_buscada"]
    
    if pd.isna(usage_key) or usage_key is None:
        print(f"[AVISO] No se encontró usageKey válido para '{especie_buscada}'. Se omite.")
        return pd.DataFrame(columns=COLUMNAS_DARWIN_CORE)
    
    url_search = "https://api.gbif.org/v1/occurrence/search"
    registros_procesados = []
    
    for pais in paises:
        offset = 0
        descargados_pais = 0
        
        while descargados_pais < max_por_pais:
            limit = min(TAMANO_PAGINA, max_por_pais - descargados_pais)
            params = {
                "taxonKey": int(usage_key),
                "country": pais,
                "hasCoordinate": "true",
                "hasGeospatialIssue": "false",
                "limit": limit,
                "offset": offset,
            }
            
            resp = requests.get(url_search, params=params, headers=HEADERS_HTTP, timeout=25)
            resp.raise_for_status()
            payload = resp.json()
            resultados = payload.get("results", [])
            
            if not resultados:
                break
            
            for reg in resultados:
                gbif_id = reg.get("gbifID")
                fila = {
                    "especie_buscada": especie_buscada,
                    "gbifID": gbif_id,
                    "url_ocurrencia_gbif": f"https://www.gbif.org/occurrence/{gbif_id}" if gbif_id else "",
                    "media_urls": extraer_urls_multimedia(reg),
                }
                for col in COLUMNAS_DARWIN_CORE:
                    if col not in fila:
                        fila[col] = reg.get(col)
                registros_procesados.append(fila)
            
            descargados_pais += len(resultados)
            offset += len(resultados)
            
            if payload.get("endOfRecords", True):
                break
            time.sleep(0.15)
            
        print(f"  - {especie_buscada} [{pais}]: {descargados_pais} registros descargados.")
    
    if not registros_procesados:
        return pd.DataFrame(columns=COLUMNAS_DARWIN_CORE)
    
    df = pd.DataFrame(registros_procesados)[COLUMNAS_DARWIN_CORE]
    
    # Limpieza de tipos numéricos y coordenadas válidas
    df["decimalLatitude"] = pd.to_numeric(df["decimalLatitude"], errors="coerce")
    df["decimalLongitude"] = pd.to_numeric(df["decimalLongitude"], errors="coerce")
    df["coordinateUncertaintyInMeters"] = pd.to_numeric(
        df["coordinateUncertaintyInMeters"], errors="coerce"
    )
    df = df.dropna(subset=["decimalLatitude", "decimalLongitude"]).copy()
    df = df.drop_duplicates(subset=["gbifID"]).reset_index(drop=True)
    
    return df

## 3. Ejecución en Lote, Exportación de CSVs y Resumen Comparativo

Se recorre el catálogo taxonómico, se descarga y limpia cada especie, se exporta su CSV individual (`gbif_<especie>_es_pt.csv`) y se consolida un DataFrame maestro (`df_total`) para análisis comparativo.

In [4]:
dataframes_por_especie: dict[str, pd.DataFrame] = {}
lista_dfs_consolidados: list[pd.DataFrame] = []

for taxon in catalogo_taxones:
    especie = taxon["especie_buscada"]
    slug = taxon["slug"]
    print(f"Procesando: {especie} (usageKey={taxon['usageKey']})...")
    
    df_esp = descargar_ocurrencias_taxon(taxon)
    dataframes_por_especie[especie] = df_esp
    
    if not df_esp.empty:
        ruta_csv = CARPETA_SALIDA / f"gbif_{slug}_es_pt.csv"
        df_esp.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
        lista_dfs_consolidados.append(df_esp)
        print(f"  -> Guardado: {ruta_csv} ({len(df_esp)} filas válidas)\n")
    else:
        print(f"  -> Sin registros georreferenciados para {especie}.\n")

if lista_dfs_consolidados:
    df_total = pd.concat(lista_dfs_consolidados, ignore_index=True)
    ruta_csv_total = CARPETA_SALIDA / "gbif_multiespecie_es_pt.csv"
    df_total.to_csv(ruta_csv_total, index=False, encoding="utf-8-sig")
    print(f"Archivo maestro consolidado guardado en: {ruta_csv_total} ({len(df_total)} registros totales)")
else:
    df_total = pd.DataFrame(columns=COLUMNAS_DARWIN_CORE)
    print("No se obtuvieron registros para ninguna especie de la lista.")

# Tabla resumen comparativa por especie y país
if not df_total.empty:
    resumen = (
        df_total.groupby(["especie_buscada", "countryCode"], dropna=False)
        .agg(
            total_registros=("gbifID", "count"),
            con_multimedia=("media_urls", lambda s: (s.fillna("") != "").sum()),
            incertidumbre_mediana_m=("coordinateUncertaintyInMeters", "median"),
            anio_min=("year", "min"),
            anio_max=("year", "max"),
        )
        .reset_index()
    )
    display(resumen)

Procesando: Canis lupus (usageKey=5219173)...
  - Canis lupus [ES]: 600 registros descargados.
  - Canis lupus [PT]: 600 registros descargados.
  -> Guardado: c:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\gbif_canis_lupus_es_pt.csv (1200 filas válidas)

Procesando: Lynx pardinus (usageKey=2435261)...
  - Lynx pardinus [ES]: 600 registros descargados.
  - Lynx pardinus [PT]: 600 registros descargados.
  -> Guardado: c:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\gbif_lynx_pardinus_es_pt.csv (1200 filas válidas)

Procesando: Ursus arctos (usageKey=2433433)...
  - Ursus arctos [ES]: 600 registros descargados.
  - Ursus arctos [PT]: 0 registros descargados.
  -> Guardado: c:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\gbif_ursus_arctos_es_pt.csv (600 filas válidas)

Procesando: Gypaetus barbatus (usageKey=2480649)...
  - Gypaetus barbatus [ES]: 600 registros descargados.
  - Gypaetus barbatus [PT]: 57 registros descargados.
  -> Guarda

,especie_buscada,countryCode,total_registros,con_multimedia,incertidumbre_mediana_m,anio_min,anio_max
0,Canis lupus,ES,600,447,50.0,2025.0,2026.0
1,Canis lupus,PT,600,520,61.0,2015.0,2026.0
2,Gypaetus barbatus,ES,600,250,12.0,2025.0,2026.0
3,Gypaetus barbatus,PT,57,0,NaN,NaN,NaN
4,Lynx pardinus,ES,600,332,28270.0,2010.0,2026.0
5,Lynx pardinus,PT,600,32,530.0,1944.0,2026.0
6,Ursus arctos,ES,600,320,27537.0,2007.0,2026.0


## 4. Cartografía Interactiva Multi-Especie (`Folium`)

Para evitar problemas con claves de API externas, todos los mapas utilizan teselas abiertas (**OpenStreetMap** y **Esri World Imagery**). Se generan:
1. **Un mapa HTML individual por especie** con agrupación dinámica (`MarkerCluster`) y popups enriquecidos (taxón, fecha, base del registro, incertidumbre espacial y enlace directo a GBIF).
2. **Un mapa comparativo multi-especie (`mapa_comparativo_multiespecie_es_pt.html`)** donde cada especie cuenta con su propia capa activable/desactivable (`folium.FeatureGroup` + `folium.LayerControl`) y un color distintivo.

In [5]:
import sys
from pathlib import Path as _PathEnlaces
_raiz_enlaces = _PathEnlaces.cwd() if (_PathEnlaces.cwd() / "src").is_dir() else _PathEnlaces.cwd().parent
if str(_raiz_enlaces / "src") not in sys.path:
    sys.path.insert(0, str(_raiz_enlaces / "src"))
from mapa_enlaces import activar_enlaces_externos

PALETA_COLORES = [
    "darkred",
    "darkblue",
    "darkgreen",
    "purple",
    "orange",
    "cadetblue",
    "darkpurple",
    "pink",
]


def crear_popup_html(fila: pd.Series) -> folium.Popup:
    """Construye un popup HTML limpio con enlace directo a la ocurrencia en GBIF."""
    fecha = fila.get("eventDate") if pd.notna(fila.get("eventDate")) else "Sin fecha"
    incertidumbre = (
        f"{int(fila['coordinateUncertaintyInMeters'])} m"
        if pd.notna(fila.get("coordinateUncertaintyInMeters"))
        else "No especificada"
    )
    basis = fila.get("basisOfRecord") if pd.notna(fila.get("basisOfRecord")) else "N/D"
    url_gbif = fila.get("url_ocurrencia_gbif", "")
    
    media_html = ""
    media_str = str(fila.get("media_urls") or "").strip()
    if media_str:
        primera_img = media_str.split(" | ")[0]
        media_html = f'<br><a href="{primera_img}" target="_blank">Ver evidencia multimedia</a>'

    html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; min-width: 220px;">
        <b>{fila.get('scientificName', fila.get('especie_buscada'))}</b><br>
        <b>País:</b> {fila.get('countryCode', 'N/D')}<br>
        <b>Fecha:</b> {fecha}<br>
        <b>Tipo registro:</b> {basis}<br>
        <b>Incertidumbre:</b> {incertidumbre}<br>
        <a href="{url_gbif}" target="_blank">Abrir ficha en GBIF ({fila.get('gbifID')})</a>
        {media_html}
    </div>
    """
    return folium.Popup(html, max_width=300)


def agregar_capas_base(mapa: folium.Map) -> None:
    """Añade capas base abiertas sin requerir API key."""
    activar_enlaces_externos(mapa)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satélite (Esri)",
        overlay=False,
        control=True,
    ).add_to(mapa)


# 4.1 Mapas individuales por especie
for idx, (especie, df_esp) in enumerate(dataframes_por_especie.items()):
    if df_esp.empty:
        continue
    
    slug = normalizar_slug(especie)
    color = PALETA_COLORES[idx % len(PALETA_COLORES)]
    centro_lat = df_esp["decimalLatitude"].mean()
    centro_lon = df_esp["decimalLongitude"].mean()
    
    mapa_ind = folium.Map(
        location=[centro_lat, centro_lon],
        zoom_start=6,
        tiles="OpenStreetMap",
    )
    agregar_capas_base(mapa_ind)
    
    cluster = MarkerCluster(name=f"{especie} ({len(df_esp)})").add_to(mapa_ind)
    for _, fila in df_esp.iterrows():
        folium.Marker(
            location=[fila["decimalLatitude"], fila["decimalLongitude"]],
            popup=crear_popup_html(fila),
            tooltip=f"{especie} ({fila['countryCode']})",
            icon=folium.Icon(color=color, icon="paw", prefix="fa"),
        ).add_to(cluster)
    
    folium.LayerControl(collapsed=False).add_to(mapa_ind)
    ruta_mapa = CARPETA_SALIDA / f"mapa_{slug}_es_pt.html"
    mapa_ind.save(str(ruta_mapa))
    print(f"Mapa individual guardado: {ruta_mapa}")


# 4.2 Mapa comparativo unificado con una capa por especie
if not df_total.empty:
    mapa_comparativo = folium.Map(
        location=[40.2, -4.5],  # Centro aproximado Península Ibérica
        zoom_start=6,
        tiles="OpenStreetMap",
    )
    agregar_capas_base(mapa_comparativo)
    
    for idx, (especie, df_esp) in enumerate(dataframes_por_especie.items()):
        if df_esp.empty:
            continue
        color = PALETA_COLORES[idx % len(PALETA_COLORES)]
        capa_especie = folium.FeatureGroup(name=f"{especie} ({len(df_esp)})", show=True)
        cluster_esp = MarkerCluster().add_to(capa_especie)
        
        for _, fila in df_esp.iterrows():
            folium.CircleMarker(
                location=[fila["decimalLatitude"], fila["decimalLongitude"]],
                radius=6,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.75,
                popup=crear_popup_html(fila),
                tooltip=f"{especie} | {fila.get('eventDate', 's/f')}",
            ).add_to(cluster_esp)
            
        capa_especie.add_to(mapa_comparativo)
    
    folium.LayerControl(collapsed=False).add_to(mapa_comparativo)
    ruta_mapa_comp = CARPETA_SALIDA / "mapa_comparativo_multiespecie_es_pt.html"
    mapa_comparativo.save(str(ruta_mapa_comp))
    print(f"\nMapa comparativo multi-especie guardado: {ruta_mapa_comp}")
    display(mapa_comparativo)

Mapa individual guardado: c:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\mapa_canis_lupus_es_pt.html
Mapa individual guardado: c:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\mapa_lynx_pardinus_es_pt.html
Mapa individual guardado: c:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\mapa_ursus_arctos_es_pt.html
Mapa individual guardado: c:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\mapa_gypaetus_barbatus_es_pt.html

Mapa comparativo multi-especie guardado: c:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\mapa_comparativo_multiespecie_es_pt.html
